# Notebook 1 — Data Cleaning and Translation

**Project:** Kosovo Labor Market Demand Analysis 2025  
**Data source:** SuperPuna.com  
**Description:** This notebook loads the raw SuperPuna job postings dataset, normalizes Albanian job titles by removing gendered suffixes, translates unique titles from Albanian to English using the Google Translate free tier, and saves the cleaned dataset to Google Drive for use in Notebook 2.

**Output:** `superpuna_cleaned.csv` saved to Google Drive  
**Next step:** Run `02_esco_standardization.ipynb`

---

## 1. Install and Import Libraries

In [1]:
# Install external libraries not available by default in Colab
!pip install deep-translator tqdm openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.8 MB/s eta 0:00:00


In [2]:
# Standard library
import re
import os
import time

# Data
import pandas as pd

# Translation
from deep_translator import GoogleTranslator
from tqdm.notebook import tqdm

# Colab utilities
from google.colab import files, drive

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Constants

In [3]:
# Column names — defined once here so any change propagates through the notebook
TITLE_COL    = 'Titulli_i_vendit_te_punes'
VACANCY_COL  = 'Numri_i_vendeve_te_lira'
COMPANY_COL  = 'Kompania'
CITY_COL     = 'Vendi_i_punes'
SALARY_COL   = 'Paga'
DATE_COL     = 'Data_e_shpalljes'

# Translation settings
# Sleep between requests avoids hitting Google Translate rate limits
TRANSLATION_SLEEP   = 0.5
CHECKPOINT_INTERVAL = 500
SOURCE_LANGUAGE     = 'sq'   # ISO 639-1 code for Albanian
TARGET_LANGUAGE     = 'en'

# Google Drive path — all intermediate files saved here
DRIVE_DIR = '/content/drive/MyDrive/kosovo_lm'

print('Constants defined.')

Constants defined.


## 3. Mount Google Drive

In [4]:
# Mount Google Drive so files persist between sessions
drive.mount('/content/drive')

os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Google Drive mounted. Working directory: {DRIVE_DIR}')

Mounted at /content/drive
Google Drive mounted. Working directory: /content/drive/MyDrive/kosovo_lm


## 4. Load Dataset

In [5]:
# Upload the raw SuperPuna dataset
# Accepts both CSV and Excel formats
print('Upload your SuperPuna dataset (CSV or Excel)...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith('.xlsx'):
    df = pd.read_excel(filename)
elif filename.endswith('.csv'):
    loaded = False
    # Try common encodings and separators — scraped CSVs are often inconsistent
    for encoding in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(filename, sep=sep, encoding=encoding,
                                 on_bad_lines='skip', engine='python')
                if df.shape[1] >= 3:
                    print(f'Loaded with encoding={encoding}, separator="{sep}"')
                    loaded = True
                    break
            except Exception:
                continue
        if loaded:
            break

# Parse date and salary columns
df[DATE_COL]   = pd.to_datetime(df[DATE_COL], errors='coerce')
df[SALARY_COL] = pd.to_numeric(df[SALARY_COL], errors='coerce')
df[VACANCY_COL]= pd.to_numeric(df[VACANCY_COL], errors='coerce').fillna(1)

print(f'Dataset loaded: {len(df):,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head(3)

Upload your SuperPuna dataset (CSV or Excel)...


Saving SuperPuna-01-01-31-12-2025.xlsx to SuperPuna-01-01-31-12-2025.xlsx
Dataset loaded: 15,422 rows x 15 columns
Columns: ['ID', 'Titulli_i_vendit_te_punes', 'Numri_i_vendeve_te_lira', 'Kompania', 'Vendi_i_punes', 'Orari_i_punes', 'Paga', 'Pershkrimi_i_punes_se_pozites', 'Pershkrimi_i_kompanise', 'Pershkrimi_i_rolit', 'Kerkest', 'Pergjegjesite', 'Kualifikimet', 'Benefitet', 'Data_e_shpalljes']


,ID,Titulli_i_vendit_te_punes,Numri_i_vendeve_te_lira,Kompania,Vendi_i_punes,Orari_i_punes,Paga,Pershkrimi_i_punes_se_pozites,Pershkrimi_i_kompanise,Pershkrimi_i_rolit,Kerkest,Pergjegjesite,Kualifikimet,Benefitet,Data_e_shpalljes
0,1,Arkatar / E,1,SuperSmokIN SH.P.K.,Prizren,10:00:00 - 15:00:00,500.0,Nëse jeni të interesuar për këtë rol klikoni A...,Tregtia me pakicë në dyqane jo të specializuar...,Si arkatar/e në SuperSmokIN SH.P.K.; do të jen...,Të keni mbi 18 vjet. Aftësi të shkëlqyera në k...,Menaxhimi i arkës dhe procesimi i transaksione...,Përvojë e mëparshme si arkatar/e ose në pozita...,Pagë konkurruese. Ambient pune miqësor dhe bas...,2025-01-01
1,2,Menaxher/e,1,Leonard A. Tahiraj B.I.,Skënderaj,08:00:00 - 16:00:00,350.0,Menaxher/e e rrjeteve sociale,NaN,Menaxher/e e rrjeteve sociale,Menaxher/e e rrjeteve sociale,Menaxher/e e rrjeteve sociale,Shollim i mesem,Menaxher/e e rrjeteve sociale,2025-01-01
2,3,Zyrtar/e për Administratë dhe Financa,1,Premium Logistics L.L.C.,Prizren,09:00:00 - 16:00:00,370.0,Detyrat dhe përgjegjësitë: • Faturimi i klient...,Kompani ne sektorin e logjistikes dhe transpor...,Detyrat dhe përgjegjësitë: • Faturimi i klient...,Të ketë përfunduar nivelin Bachelor; drejtimi ...,Detyrat dhe përgjegjësitë: • Faturimi i klient...,NaN,NaN,2025-01-01


## 5. Normalize Albanian Job Titles

In [6]:
def normalize_albanian_title(title):
    """
    Normalize an Albanian job title to remove gendered variants
    and diacritic characters that create artificial duplicates.

    Albanian uses both slash notation (Shites/e) and backslash
    notation (Shites\\e) to indicate male/female variants of the
    same role. These are collapsed to a single form before translation
    to reduce the number of unique titles that need to be processed.

    Examples:
        'Kamarier/e'  -> 'Kamarier'
        'Shites\\e'   -> 'Shites'
        'Arkatar / E' -> 'Arkatar'
        'Infermier(e)'-> 'Infermier'
    """
    if not isinstance(title, str):
        return ''
    t = title.strip()
    # Normalize backslash to forward slash before applying regex
    t = t.replace('\\', '/')
    # Remove gender slash patterns: /e, /ja, /i, / E etc.
    t = re.sub(r'\s*/\s*[eEjaJAiI]{1,2}\b', '', t)
    # Remove trailing slash with nothing after
    t = re.sub(r'\s*/\s*$', '', t)
    # Remove parenthetical gender markers: (e), (i), (ja)
    t = re.sub(r'\s*\([eEiIjJ]{1,2}\)', '', t)
    # Standardize Albanian diacritics
    t = t.replace('e', 'e').replace('E', 'E')
    t = t.replace('c', 'c').replace('C', 'C')
    t = re.sub(r'\s+', ' ', t).strip()
    return t


df['title_normalized'] = df[TITLE_COL].apply(normalize_albanian_title)

before = df[TITLE_COL].nunique()
after  = df['title_normalized'].nunique()

print('Title normalization complete.')
print(f'Unique titles before normalization : {before:,}')
print(f'Unique titles after normalization  : {after:,}')
print(f'Variants removed                   : {before - after:,}')

Title normalization complete.
Unique titles before normalization : 6,649
Unique titles after normalization  : 6,410
Variants removed                   : 239


## 6. Detect Language — Separate Albanian and English Titles

In [7]:
def is_likely_english(text):
    """
    Heuristic check to identify titles already written in English.
    Titles identified as English are excluded from translation to
    reduce API calls and avoid introducing translation errors.
    """
    if not isinstance(text, str):
        return False
    english_keywords = [
        'manager', 'engineer', 'developer', 'analyst', 'designer',
        'coordinator', 'specialist', 'consultant', 'director', 'officer',
        'assistant', 'supervisor', 'technician', 'accountant', 'sales',
        'marketing', 'finance', 'hr ', 'it ', 'ceo', 'cfo', 'cto',
        'operator', 'intern', 'senior', 'junior', 'lead', 'head of',
        'driver', 'nurse', 'teacher', 'chef', 'cook', 'security'
    ]
    return any(kw in text.lower() for kw in english_keywords)


unique_titles     = [t for t in df['title_normalized'].dropna().unique() if t.strip()]
already_english   = [t for t in unique_titles if is_likely_english(t)]
needs_translation = [t for t in unique_titles if not is_likely_english(t)]

print(f'Total unique normalized titles : {len(unique_titles):,}')
print(f'Already English (skip)         : {len(already_english):,}')
print(f'Requires translation           : {len(needs_translation):,}')

Total unique normalized titles : 6,410
Already English (skip)         : 828
Requires translation           : 5,582


## 7. Translate Albanian Titles to English

In [11]:
def translate_with_retry(translator, title, retries=3):
    """
    Translate a single title with exponential backoff on failure.
    Falls back to the original title if all retries are exhausted.
    """
    for attempt in range(retries):
        try:
            result = translator.translate(title)
            return result if result else title
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return title


def translate_batch(titles, src=SOURCE_LANGUAGE, target=TARGET_LANGUAGE,
                    batch_size=50):
    """
    Translate a list of titles using the Google Translate free tier.
    Saves a checkpoint to Google Drive every CHECKPOINT_INTERVAL titles
    so progress is not lost if the session disconnects.
    """
    translation_map = {}
    translator = GoogleTranslator(source=src, target=target)

    for i in tqdm(range(0, len(titles), batch_size), desc="Translating"):
        batch = titles[i:i+batch_size]
        for title in batch:
            translation_map[title] = translate_with_retry(translator, title)

        if len(translation_map) % CHECKPOINT_INTERVAL == 0:
            pd.DataFrame(translation_map.items(),
                         columns=["albanian_title", "english_title"]
                        ).to_csv(f"{DRIVE_DIR}/translation_map.csv", index=False)
            print(f"Checkpoint saved: {len(translation_map):,} titles translated.")

        time.sleep(TRANSLATION_SLEEP)

    return translation_map


# ── Load or generate translation map ─────────────────────────────────────────
# Set TRANSLATION_MODE before running this cell:
#
#   "upload"    — upload translation_map.csv manually from your computer
#                 (download it from the repository data/ folder first)
#
#   "drive"     — load translation_map.csv from Google Drive
#                 (if you have run this notebook before)
#
#   "translate" — run full translation from scratch (takes ~45 minutes)

TRANSLATION_MODE = "upload"  # change to "drive" or "translate" if needed

checkpoint_path = f"{DRIVE_DIR}/translation_map.csv"
translation_map = {}

if TRANSLATION_MODE == "upload":
    print("Upload translation_map.csv from your computer...")
    print("You can download it from the repository: data/translation_map.csv\n")
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        saved = pd.read_csv(fname)
        translation_map = dict(zip(saved["albanian_title"], saved["english_title"]))
        # Save to Drive so future runs can use "drive" mode
        saved.to_csv(checkpoint_path, index=False)
        print(f"Translation map loaded: {len(translation_map):,} titles.")
        print("Translation step skipped.")
    else:
        print("No file uploaded. Change TRANSLATION_MODE to 'translate' to run from scratch.")

elif TRANSLATION_MODE == "drive":
    if os.path.exists(checkpoint_path):
        saved = pd.read_csv(checkpoint_path)
        translation_map = dict(zip(saved["albanian_title"], saved["english_title"]))
        print(f"Translation map loaded from Google Drive: {len(translation_map):,} titles.")
        print("Translation step skipped.")
    else:
        print(f"No file found at {checkpoint_path}")
        print("Change TRANSLATION_MODE to 'upload' or 'translate'.")

elif TRANSLATION_MODE == "translate":
    print(f"Starting translation of {len(needs_translation):,} titles...")
    print(f"Estimated time: approximately 45 minutes.")
    print(f"Checkpoints saved every {CHECKPOINT_INTERVAL} titles to Google Drive.\n")

    translation_map = translate_batch(needs_translation)

    for t in already_english:
        translation_map[t] = t

    pd.DataFrame(translation_map.items(),
                 columns=["albanian_title", "english_title"]
                ).to_csv(checkpoint_path, index=False)

    print(f"\nTranslation complete: {len(translation_map):,} titles.")
    print(f"Translation map saved to: {checkpoint_path}")

else:
    print(f"Unknown TRANSLATION_MODE: '{TRANSLATION_MODE}'")
    print("Valid options are: 'upload', 'drive', 'translate'")

Upload translation_map.csv from your computer...
You can download it from the repository: data/translation_map.csv



Saving translation_map.csv to translation_map (1).csv
Translation map loaded: 6,172 titles.
Translation step skipped.


## 8. Manual Translation Corrections

In [12]:
# Some short Albanian words are mistranslated out of context.
# These corrections are applied after automatic translation.
# Add additional entries as needed after reviewing translation output.
manual_fixes = {
    'Workplace'        : 'Worker',
    'paymaster'        : 'Cashier',
    'steward'          : 'Waiter',
    'Steward'          : 'Waiter',
    'Agent'            : 'Field Agent',
}

fixed_count = 0
for albanian, english in translation_map.items():
    if english in manual_fixes:
        translation_map[albanian] = manual_fixes[english]
        fixed_count += 1

print(f'Manual corrections applied: {fixed_count}')

# Save corrected map
pd.DataFrame(translation_map.items(),
             columns=['albanian', 'english']
            ).to_csv(f'{DRIVE_DIR}/translation_map.csv', index=False)

Manual corrections applied: 0


## 9. Add English Titles to Dataset

In [13]:
df['title_english'] = df['title_normalized'].map(translation_map)
# Where no translation exists, retain the normalized Albanian title
df['title_english'] = df['title_english'].fillna(df['title_normalized'])

print('English titles added to dataset.')
print(f'Rows with English title: {df["title_english"].notna().sum():,}')
df[[TITLE_COL, 'title_normalized', 'title_english']].head(10)

English titles added to dataset.
Rows with English title: 15,422


,Titulli_i_vendit_te_punes,title_normalized,title_english
0,Arkatar / E,Arkatar,Cashier
1,Menaxher/e,Menaxher,manager
2,Zyrtar/e për Administratë dhe Financa,Zyrtar për Administratë dhe Financa,Zyrtar për Administratë dhe Financa
3,Gazetar/e,Gazetar,Journalist
4,Menaxher/e per rrjete sociale,Menaxher per rrjete sociale,Manager for social networks
5,Operations Manager,Operations Manager,Operations Manager
6,Asistent i Teknologjisë së Informacionit,Asistent i Teknologjisë së Informacionit,Asistent i Teknologjisë së Informacionit
7,Puntore,Puntore,Worker
8,R&D Technician and IT Specialist,R&D Technician and IT Specialist,R&D Technician and IT Specialist
9,Asistent Teknik dhe Logjistikë,Asistent Teknik dhe Logjistikë,Asistent Teknik dhe Logjistikë


## 10. Save Cleaned Dataset to Google Drive

In [14]:
output_path = f'{DRIVE_DIR}/superpuna_cleaned.csv'
df.to_csv(output_path, index=False)

print('Cleaned dataset saved successfully.')
print(f'Path     : {output_path}')
print(f'Rows     : {len(df):,}')
print(f'Columns  : {len(df.columns)}')
print('\nNext step: open 02_esco_standardization.ipynb')

Cleaned dataset saved successfully.
Path     : /content/drive/MyDrive/kosovo_lm/superpuna_cleaned.csv
Rows     : 15,422
Columns  : 17

Next step: open 02_esco_standardization.ipynb
